# Phase 5.1: LOSO CV -- Konsep & Setup

**Tujuan:** Phase 4 memakai `StratifiedGroupKFold(n_splits=5)` (grouped per subjek, subject-independent,
5 fold). Phase 5 menguji ulang dengan **Leave-One-Subject-Out (LOSO)** -- setiap fold menyisakan
**tepat 1 subjek** sebagai test set (14 subjek untuk training, 1 subjek untuk testing), diulang 15
kali (sebanyak jumlah subjek di SEED). Ini adalah protokol evaluasi gold-standard di literatur EEG
subject-independent karena:

1. **Lebih ketat** dari 5-fold grouped: tiap subjek pernah jadi test set sendiri (bukan cuma 1/5
   subjek per fold), sehingga variansi generalisasi antar-individu lebih transparan.
2. **Dapat dibandingkan langsung** dengan penelitian EEG lain yang mayoritas melaporkan LOSO,
   bukan k-fold biasa.
3. Memungkinkan analisis **per-subjek** (5.3) -- subjek mana yang sinyalnya mudah/sulit
   digeneralisasi ke model yang dilatih dari subjek lain.

**Dataset yang diuji** (5 fitur kunci, konsisten dengan hasil terbaik Phase 3-4):
`GC`, `PDC` (band terbaik berdasar F-score), `Spectral`, `Combined` (semua per-channel),
`GC+PDC_ROI+Spectral` (fitur ROI terbaik dari Phase 4, 75.56% akurasi grouped-CV).

Notebook ini (5.1) hanya melakukan **setup & sanity check** struktur data + LOSO split.
Perhitungan LOSO CV penuh (semua model) ada di **5.2**.

In [1]:
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.facecolor'] = 'white'

PHASE3_CSV_DIR = r"D:\Skripsi\new_data\phase_3_feature_engineering\csv"
ROI_DIR = r"D:\Skripsi\new_data\phase_3_feature_engineering\3.4_roi_aggregation\output"
TUNING_PARAMS_CSV = r"D:\Skripsi\new_data\phase_4_classification\4.6_hyperparameter_tuning\output\tuning_best_params.csv"
BASE = r"D:\Skripsi\new_data\phase_5_loso"
KEY_COLS = ['subject_id', 'subject_num', 'session', 'trial', 'class', 'class_label']

def _feature_cols(df):
    exclude = KEY_COLS + [c for c in df.columns if 'hub_channel' in c]
    cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['float64', 'int64']]
    return [c for c in cols if df[c].std() > 1e-10]

print("Setup OK.")

Setup OK.


In [2]:
from sklearn.feature_selection import f_classif

df_gc = pd.read_csv(os.path.join(PHASE3_CSV_DIR, "engineered_features_gc.csv"))
df_combined = pd.read_csv(os.path.join(PHASE3_CSV_DIR, "engineered_features_combined.csv"))
df_spectral = pd.read_csv(os.path.join(PHASE3_CSV_DIR, "engineered_features_spectral.csv"))
df_roi_all = pd.read_csv(os.path.join(ROI_DIR, "engineered_features_roi_all.csv"))  # GC_ROI+PDC_ROI(6-band)+Spectral

# Best PDC band by mean ANOVA F-score (konsisten dengan pola Phase 4)
pdc_files = [f for f in os.listdir(PHASE3_CSV_DIR)
             if f.startswith("engineered_features_pdc_") and "broadband" not in f]
best_pdc, best_pdc_name, best_score = None, None, -1
for f in pdc_files:
    df_pdc = pd.read_csv(os.path.join(PHASE3_CSV_DIR, f))
    band = f.replace("engineered_features_pdc_", "").replace(".csv", "")
    feat_cols = _feature_cols(df_pdc)
    f_scores, _ = f_classif(df_pdc[feat_cols].fillna(0).values, df_pdc['class'].values)
    score = np.nanmean(f_scores)
    if score > best_score:
        best_score, best_pdc, best_pdc_name = score, df_pdc, f"PDC_{band.title()}"

DATASETS_RAW = {
    'GC': df_gc,
    best_pdc_name: best_pdc,
    'Spectral': df_spectral,
    'Combined': df_combined,
    'GC+PDC_ROI+Spectral': df_roi_all,
}

def prepare_features(df, normalize=True):
    feature_cols = _feature_cols(df)
    df_use = df.copy()
    if normalize:
        grp = df_use.groupby('subject_id')[feature_cols]
        mean = grp.transform('mean')
        std = grp.transform('std').replace(0, np.nan)
        df_use[feature_cols] = (df_use[feature_cols] - mean) / std
    X = df_use[feature_cols].fillna(0).values
    y = df['class'].values
    groups = df['subject_num'].values
    return X, y, groups, feature_cols

datasets = {}
for name, df in DATASETS_RAW.items():
    X, y, groups, features = prepare_features(df, normalize=True)
    datasets[name] = {'X': X, 'y': y, 'groups': groups, 'features': features, 'df': df}
    print(f"{name:24s}: X={X.shape}, subjects={len(np.unique(groups))}, trials/subject={len(y)//len(np.unique(groups))}")

GC                      : X=(675, 282), subjects=15, trials/subject=45


PDC_Theta               : X=(675, 282), subjects=15, trials/subject=45


Spectral                : X=(675, 640), subjects=15, trials/subject=45


Combined                : X=(675, 2614), subjects=15, trials/subject=45


GC+PDC_ROI+Spectral     : X=(675, 990), subjects=15, trials/subject=45


## Sanity Check: Struktur LOSO Split

In [3]:
from sklearn.model_selection import LeaveOneGroupOut

logo = LeaveOneGroupOut()
X_demo, y_demo, groups_demo = datasets['GC']['X'], datasets['GC']['y'], datasets['GC']['groups']
n_folds = logo.get_n_splits(groups=groups_demo)
print(f"Jumlah fold LOSO: {n_folds} (= jumlah subjek)")
print()
print(f"{'Fold':>5s} {'Held-out subject':>18s} {'N_train':>10s} {'N_test':>8s}")
for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_demo, y_demo, groups_demo)):
    held = int(np.unique(groups_demo[test_idx])[0])
    print(f"{fold_idx:>5d} {('S' + str(held).zfill(2)):>18s} {len(train_idx):>10d} {len(test_idx):>8d}")

assert n_folds == 15, "Harus 15 fold (15 subjek di dataset SEED)"
print("\nOK: setiap fold test-nya tepat 1 subjek penuh (45 trial), tidak ada subjek yang")
print("bocor antara train dan test (dijamin oleh LeaveOneGroupOut + groups=subject_num).")

Jumlah fold LOSO: 15 (= jumlah subjek)

 Fold   Held-out subject    N_train   N_test
    0                S01        630       45
    1                S02        630       45
    2                S03        630       45
    3                S04        630       45
    4                S05        630       45
    5                S06        630       45
    6                S07        630       45
    7                S08        630       45
    8                S09        630       45
    9                S10        630       45
   10                S11        630       45
   11                S12        630       45
   12                S13        630       45
   13                S14        630       45
   14                S15        630       45

OK: setiap fold test-nya tepat 1 subjek penuh (45 trial), tidak ada subjek yang
bocor antara train dan test (dijamin oleh LeaveOneGroupOut + groups=subject_num).


**Kesimpulan 5.1:** Struktur data dan LOSO split sudah tervalidasi -- 5 dataset kunci siap,
15 fold LOSO per dataset (1 subjek/fold, 45 trial held-out), tidak ada leakage subjek.
Lanjut ke **5.2** untuk menjalankan LOSO CV penuh (5 dataset x 4 model dasar + Voting_Soft).